# Topic: ML: Feature Engineering

## Definition (30-second explanation)
* Feature engineering is the process of using domain knowledge to create, transform, extract, or bin variables from raw data.
* Its goal is to highlight the most informative signals for the machine learning algorithm to learn from.
* It bridges the gap between raw data and predictive modeling, often providing a higher performance boost than algorithm tuning.

## Why Interviewers Ask This
* To test your **business acumen**: Can you translate a raw data point (e.g., birthdate) into a meaningful business metric (e.g., age group)?
* To check for **data leakage awareness**: Do you know to apply transformations *after* the train-test split?
* To evaluate your **practical data manipulation skills**: Can you write efficient Pandas/SQL code to transform variables?

## Core Concepts
* **Feature Extraction:** Pulling specific elements from complex data (e.g., extracting 'day of week' or 'month' from a datetime column).
* **Feature Transformation:** Applying mathematical functions to normalize or un-skew data (e.g., Log transformation).
* **Binning:** Converting continuous numerical variables into discrete categorical buckets (e.g., Age 0-18 = 'Young').
* **Interaction Features:** Combining two or more features to capture their joint effect (e.g., `revenue / orders` = `revenue_per_order`).

## When to Use
* **Heavily skewed data:** Apply log transformations to normalize right-skewed financial or behavioral data.
* **Complex raw fields:** When dealing with raw datetimes, raw text, or JSON fields that algorithms cannot natively process.
* **Non-linear relationships:** Use binning or polynomial features to help simple linear models capture non-linear patterns.

## Advantages
* Significantly improves model accuracy and interpretability.
* Reduces model complexity by providing the algorithm with pre-calculated, highly predictive signals.
* Forces the data scientist to deeply understand the business context of the data.

## Limitations
* Over-engineering can lead to the "curse of dimensionality" and over-fitting.
* Increases the complexity of the data pipeline and deployment (every feature must be replicated exactly in production).
* High risk of data leakage if aggregations or scaling are calculated across the entire dataset before splitting.

## Common Comparisons
* **Feature Engineering vs. Feature Selection:** Engineering *creates* new information or representations from existing data; Selection *filters* down the features to keep only the most useful ones.
* **Binning vs. Scaling:** Binning discretizes continuous data into groups; Scaling (MinMax/Standardization) keeps data continuous but changes its range.

## Common Interview Traps
* **Data Leakage:** Computing global statistics (like mean for imputation or target encoding) on the *entire* dataset before doing a train/test split.
* **Creating meaningless features:** Engineering complex mathematical interactions that have absolutely no business logic or interpretability.
* **Ignoring division by zero:** Failing to account for zero denominators when creating ratio features (like `clicks / impressions`).

## Python / SQL Syntax (if applicable)
* **Date Extraction:** `df['month'] = df['date_col'].dt.month`
* **Log Transform:** `df['log_rev'] = np.log1p(df['revenue'])` *(Note: log1p safely handles zeros)*
* **Binning:** `df['age_group'] = pd.cut(df['age'], bins=[0, 18, 65, 100], labels=['Youth', 'Adult', 'Senior'])`
* **Ratio creation safely:** `df['conversion_rate'] = np.where(df['impressions'] == 0, 0, df['clicks'] / df['impressions'])`

## 45-Second Interview Answer
"Feature engineering is arguably the most impactful step in the ML lifecycle. It involves using domain knowledge to extract, transform, and create new variables from raw data—such as extracting the day of the week from a timestamp, binning ages, or creating interaction ratios like revenue-per-user. The most critical rule is to always build features that make business sense, and to apply stateful transformations *strictly after* the train-test split to prevent data leakage."

## Example Questions:

### Q1. You have a raw 'datetime' column. What features can you engineer from it?
* **Ideal Interview Answer:** I can extract temporal components like year, month, day, hour, and minute. More importantly, I can engineer business-relevant cyclic features like 'day_of_week', 'is_weekend', 'is_holiday', or 'time_since_last_event' (e.g., days since last purchase).
* **Common Mistakes:** Forgetting to handle timezones. Just extracting raw numbers without considering cyclic encoding (like sine/cosine for months) for distance-based algorithms.
* **Likely Follow-up:** How would you encode 'month' for a linear regression model so that December (12) and January (1) are understood to be close to each other?

### Q2. What is the difference between feature engineering and feature selection?
* **Ideal Interview Answer:** Feature engineering is the generative process of creating new features or transforming existing ones to better represent the underlying problem (e.g., creating a BMI feature from height and weight). Feature selection is the filtering process of reducing the number of features to only the most predictive ones to prevent overfitting and reduce dimensionality.
* **Common Mistakes:** Confusing dimensionality reduction techniques (like PCA) with feature selection.
* **Likely Follow-up:** What are some techniques you would use for feature selection after you've generated hundreds of features?

### Q3. When would you use log transformation and why?
* **Ideal Interview Answer:** I use log transformations primarily on heavily right-skewed continuous variables, such as income, prices, or time-on-page. The log transform compresses the long tail, bringing the distribution closer to a normal distribution, which helps linear models and distance-based algorithms perform better and prevents extreme outliers from dominating the loss function.
* **Common Mistakes:** Using `np.log()` on data that contains zeros without adding a constant (failing to use `np.log1p()`).
* **Likely Follow-up:** If your data contains negative values, how would you handle the transformation instead of a standard log?

### Q4. How do you create interaction features and when are they useful?
* **Ideal Interview Answer:** Interaction features are created by combining two or more distinct features using mathematical operations—commonly multiplication or division. For example, dividing total revenue by total orders to get 'average_order_value'. They are incredibly useful when the combined effect of two variables has a stronger predictive power than the variables independently, effectively helping simple linear models capture non-linear relationships.
* **Common Mistakes:** Creating interaction features randomly without business logic (e.g., multiplying age by zip code).
* **Likely Follow-up:** Tree-based models like Random Forests can learn non-linear interactions natively. Do you still need to explicitly create interaction features for them?

### Q5. What is target encoding and what is the risk of using it naively?
* **Ideal Interview Answer:** Target encoding replaces a categorical value with the mean of the target variable for that category. Naively applying it across the entire dataset before splitting introduces massive data leakage, as the feature now contains direct information about the target variable from the validation/test set. It also risks severe overfitting on rare categories.
* **Common Mistakes:** Explaining the concept but forgetting to mention the primary mitigation strategies: using k-fold cross-validation or smoothing for the encoding.
* **Likely Follow-up:** How do you implement target encoding safely to prevent this data leakage?

## Practice Questions:

### Q1. Write Pandas code to engineer an 'avg_order_value' ratio feature and handle edge cases (zeros and nulls).

In [18]:
# Data:
import pandas as pd
import numpy as np

# Mock Data
data = {
    'customer_id': [1, 2, 3, 4],
    'total_revenue': [500.0, 0.0, 150.0, np.nan],
    'total_orders': [5, 0, 1, 3]
}
df = pd.DataFrame(data)

In [19]:
# Handle division by zero using np.where
df['avg_order_value'] = np.where(df['total_orders'] == 0, 
                                   0, 
                                   df['total_revenue'] / df['total_orders'])
  
# Handle nulls (assuming train_median is pre-calculated from training data only)
train_median = df['avg_order_value'].median() 
df['avg_order_value'] = df['avg_order_value'].fillna(train_median)

In [20]:
df

,customer_id,total_revenue,total_orders,avg_order_value
0,1,500.0,5,100.0
1,2,0.0,0,0.0
2,3,150.0,1,150.0
3,4,NaN,3,100.0


**Explanation:** "I use np.where to safely catch users with zero orders and set their average to 0, avoiding infinite values. For missing revenue data resulting in NaNs, I impute using the median. Crucially, in a real pipeline, I would ensure this median is calculated only on the training set to prevent data leakage."

**Common Mistakes:** Using a simple division df['total_revenue'] / df['total_orders'] which results in inf for zero denominators, breaking downstream ML algorithms. Calculating the imputation median on the entire dataset before splitting.